In [1]:
import numpy as np

%load_ext autoreload
%autoreload 2
from feature_evaluation_methods import node_variance, best_variance_reduction_for_feature, \
    feature_variance_reduction_scores
import pandas as pd


def small_dataset(n_rows=5, n_features=5, n_labels=3, seed=42):
    """
    Generate a small, reproducible binary dataset with given dimensions.

    Parameters
    ----------
    n_rows : int
        Number of samples.
    n_features : int
        Number of feature columns (f1, f2, ...).
    n_labels : int
        Number of label columns (y1, y2, ...).
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    df : pd.DataFrame
        Combined DataFrame of features + labels.
    X : pd.DataFrame
        Feature-only DataFrame.
    Y : pd.DataFrame
        Label-only DataFrame.
    """
    rng = np.random.default_rng(seed)

    # Generate binary features and labels
    X = pd.DataFrame(
        rng.integers(0, 2, size=(n_rows, n_features)),
        columns=[f"f{i + 1}" for i in range(n_features)]
    )

    Y = pd.DataFrame(
        rng.integers(0, 2, size=(n_rows, n_labels)),
        columns=[f"y{i + 1}" for i in range(n_labels)]
    )

    # Combine
    df = pd.concat([X, Y], axis=1)

    return df, X, Y


def test_node_variance_manual():
    df, X, Y = small_dataset()
    # Manually computed: total variance = 0.319
    np.testing.assert_almost_equal(node_variance(Y), 0.319, decimal=3)


def test_best_variance_reduction_f1_manual():
    df, X, Y = small_dataset()
    reduction, thr = best_variance_reduction_for_feature(X["f1"].values, Y)
    # Corrected expected value
    np.testing.assert_almost_equal(reduction, 0.3533, decimal=3)
    assert np.isclose(thr, 0.5)


def test_feature_variance_reduction_scores_best_feature():
    df, X, Y = small_dataset()
    result = feature_variance_reduction_scores(X, Y)
    assert isinstance(result, dict)
    np.testing.assert_allclose(result["best_variance_reduction"], 0.3533, rtol=1e-3)
    assert 0 < result["best_threshold"] <= 1

In [6]:
df, X, Y = small_dataset()
df

,f1,f2,f3,f4,f5,y1,y2,y3
0,0,1,1,0,0,1,0,1
1,1,0,1,0,0,1,0,0
2,1,1,1,1,1,0,0,1
3,1,1,0,1,0,1,0,1
4,1,0,0,1,1,1,0,1


In [4]:
test_node_variance_manual()
test_best_variance_reduction_f1_manual()
test_feature_variance_reduction_scores_best_feature()

AssertionError: 
Arrays are not almost equal to 3 decimals
 ACTUAL: 0.0
 DESIRED: 0.3533

In [4]:
results = []
for col in X.columns:
    reduction, thr = best_variance_reduction_for_feature(X[col].values, Y)
    results.append({
        "feature": col,
        "best_variance_reduction": round(float(reduction), 4),
        "best_threshold": thr
    })

# Convert to DataFrame for easy viewing
results_df = pd.DataFrame(results).sort_values("best_variance_reduction", ascending=False).reset_index(drop=True)

print(results_df)

  feature  best_variance_reduction  best_threshold
0      f2                   0.0867             0.5
1      f4                   0.0867             0.5
2      f3                   0.0533             0.5
3      f1                   0.0000             NaN
4      f5                   0.0000             NaN


In [13]:
import pandas as pd
from itertools import combinations

# --- Load dataset ---
df, X, Y = small_dataset(n_rows=20, n_features=5, n_labels=3)

# --- Create all possible combinations (2 to 5 features) ---
min_size = 2
max_size = 5

# Collect all new columns in a dictionary first
combined_data = {}

for r in range(min_size, max_size + 1):
    for combo in combinations(X.columns, r):
        col_name = "_".join(combo)
        combined_data[col_name] = X[list(combo)].sum(axis=1)

# --- Build DataFrame all at once ---
combined_features = pd.DataFrame(combined_data)

# --- Merge original features and combinations ---
X_all = pd.concat([X, combined_features], axis=1)

# --- Compute best variance reduction for each feature/group ---
results = []
for col in X_all.columns:
    reduction, thr = best_variance_reduction_for_feature(X_all[col].values, Y)
    results.append({
        "feature": col,
        "best_variance_reduction": round(float(reduction), 4),
        "best_threshold": thr
    })

# --- Convert to DataFrame and sort ---
results_df = (
    pd.DataFrame(results)
    .sort_values("best_variance_reduction", ascending=False)
    .reset_index(drop=True)
)

print(results_df)

           feature  best_variance_reduction  best_threshold
0            f2_f5                   0.0913             1.5
1      f2_f3_f4_f5                   0.0842             2.5
2         f1_f4_f5                   0.0746             1.5
3         f2_f3_f5                   0.0711             1.5
4         f2_f3_f4                   0.0642             1.5
5      f1_f2_f3_f5                   0.0569             2.5
6            f3_f5                   0.0569             1.5
7         f1_f2_f3                   0.0558             0.5
8               f4                   0.0537             0.5
9         f1_f3_f4                   0.0525             1.5
10     f1_f2_f3_f4                   0.0508             1.5
11           f1_f3                   0.0506             0.5
12              f3                   0.0490             0.5
13           f1_f4                   0.0475             1.5
14           f2_f3                   0.0470             0.5
15        f3_f4_f5                   0.0

In [9]:
from SIDER_dataset.libraries.XofN_library import generate_XofN_list_multi_custom, calculate_mdi_multi_rf, \
    generate_XofN_list_multi_jaccard
from SIDER_dataset.libraries.train_test_library import get_logger

max_size = 5
logger = get_logger("test_feature_evaluation_methods.logs")
features = ["f1", "f2", "f3", "f4", "f5"]
labels = ["y1", "y2", "y3"]
df, X, Y = small_dataset(n_rows=20, n_features=5, n_labels=3)

feature_rankings = calculate_mdi_multi_rf(df, Y)
print(feature_rankings.head())

XofN_groupings, avg_features = generate_XofN_list_multi_custom(
    df,
    feature_rankings,
    max_size,
    labels,
    logger
)
print(XofN_groupings)
print(len(XofN_groupings))
print(avg_features)

          y1        y2        y3  MDI_mean
f3  0.255564  0.239762  0.180616  0.225314
f5  0.184687  0.230501  0.254923  0.223370
f4  0.293005  0.176469  0.137343  0.202272
f1  0.148286  0.166159  0.248617  0.187687
f2  0.118458  0.187109  0.178501  0.161356

generate_XofN_list -> Generating groupings based on variance reduction:variance reduction.


🔄 Processing features: 100%|██████████| 5/5 [00:05<00:00,  1.19s/feat]

XofN_groups: [['f3', 'f5', 'f2', 'f4']]
XofN groups with 4 features: 1
[['f3', 'f5', 'f2', 'f4']]
1
4.0


In [7]:
scores = feature_variance_reduction_scores(X, Y)
scores

,feature,best_variance_reduction,best_threshold
f2,f2,0.086667,0.5
f4,f4,0.086667,0.5
f3,f3,0.053333,0.5
f1,f1,0.000000,NaN
f5,f5,0.000000,NaN


In [14]:
mdi_scores = calculate_mdi_multi_rf(df, Y)
mdi_scores

,y1,y2,y3,MDI_mean
f3,0.255564,0.239762,0.180616,0.225314
f5,0.184687,0.230501,0.254923,0.223370
f4,0.293005,0.176469,0.137343,0.202272
f1,0.148286,0.166159,0.248617,0.187687
f2,0.118458,0.187109,0.178501,0.161356


In [4]:
import pandas as pd
from XofN_library import eval_jac_simil, calculate_mdi_multi_rf
from SIDER_dataset.libraries.prepare_dataset_library import get_features
from SIDER_dataset.libraries.XofN_library import calculate_mdi_multi_rf, generate_XofN_list_multi_jaccard
from SIDER_dataset.libraries.train_test_library import get_logger
import itertools

max_size = 5
logger = get_logger("test_feature_evaluation_methods.logs")
features = ["f1", "f2", "f3", "f4", "f5"]
labels = ["y1", "y2", "y3"]

df, X, Y = small_dataset()
print(df)

feature_rankings = calculate_mdi_multi_rf(df, Y)
print(feature_rankings)

for fa, fb in itertools.combinations(get_features(df, Y), 2):
    print(fa, fb, ":", eval_jac_simil(df[fa], df[fb]))

   f1  f2  f3  f4  f5  y1  y2  y3
0   0   1   1   0   0   1   0   1
1   1   0   1   0   0   1   0   0
2   1   1   1   1   1   0   0   1
3   1   1   0   1   0   1   0   1
4   1   0   0   1   1   1   0   1
          y1   y2        y3  MDI_mean
f2  0.142361  0.0  0.379664  0.174008
f5  0.314032  0.0  0.165630  0.159887
f4  0.173713  0.0  0.253006  0.142240
f3  0.302696  0.0  0.097533  0.133410
f1  0.067198  0.0  0.104167  0.057121
f1 f2 : 0.4
f1 f3 : 0.4
f1 f4 : 0.75
f1 f5 : 0.5
f2 f3 : 0.5
f2 f4 : 0.5
f2 f5 : 0.25
f3 f4 : 0.19999999999999996
f3 f5 : 0.25
f4 f5 : 0.6666666666666667

generate_XofN_list -> Generating groupings based on variance reduction:variance reduction.


🔄 Processing features: 100%|██████████| 5/5 [00:05<00:00,  1.12s/feat]

XofN_groups: [['f2', 'f4', 'f1', 'f5', 'f3']]
XofN groups with 5 features: 1
[['f2', 'f4', 'f1', 'f5', 'f3']]


In [2]:
import pandas as pd
from XofN_library import eval_jac_simil, calculate_mdi_multi_rf
from SIDER_dataset.libraries.prepare_dataset_library import get_features
from SIDER_dataset.libraries.XofN_library import calculate_mdi_multi_rf, generate_XofN_list_multi_jaccard
from SIDER_dataset.libraries.train_test_library import get_logger
import itertools

df, X, Y = small_dataset(n_rows=10, n_features=14, n_labels=3)
max_size = 5
logger = get_logger("test_feature_evaluation_methods.logs")
features = get_features(df, Y)
labels = ["y1", "y2", "y3"]
print(df)

feature_rankings = calculate_mdi_multi_rf(df, Y)
print(feature_rankings)

for fa, fb in itertools.combinations(feature_rankings.index, 2):
    print(fa, fb, ":", eval_jac_simil(df[fa], df[fb]))

XofN_groupings, avg_features, time = generate_XofN_list_multi_jaccard(
    df,
    feature_rankings,
    5,
    logger
)
print(XofN_groupings)

   f1  f2  f3  f4  f5  f6  f7  f8  f9  f10  f11  f12  f13  f14  y1  y2  y3
0   0   1   1   0   0   1   0   1   0    0    1    1    1    1   1   0   0
1   1   1   1   0   1   0   1   0   0    1    1    1    0    1   0   1   1
2   1   0   0   0   0   1   1   0   1    1    0    1    0    1   0   0   1
3   1   0   0   1   0   1   1   1   1    0    0    0    0    0   0   1   0
4   1   0   1   1   1   1   0   1   0    0    1    0    0    0   1   0   0
5   1   0   0   0   1   0   0   0   1    1    1    0    0    1   1   0   1
6   1   1   0   0   1   1   0   1   1    0    1    0    0    1   1   1   1
7   1   0   1   0   1   0   1   1   1    1    0    1    0    1   1   0   0
8   1   0   1   1   0   0   0   0   0    1    1    0    1    1   0   1   0
9   0   1   1   1   1   1   0   1   1    0    1    0    0    0   0   0   0
           y1        y2        y3  MDI_mean
f8   0.135705  0.062657  0.148912  0.115758
f4   0.089082  0.076978  0.157231  0.107764
f3   0.070952  0.066424  0.158360  0.098579

🔄 Processing features: 100%|██████████| 14/14 [00:05<00:00,  2.55feat/s]

XofN_groups: [['f8', 'f6', 'f9', 'f1', 'f5'], ['f4', 'f3', 'f11', 'f2', 'f14'], ['f7', 'f12', 'f10', 'f13']]
XofN groups with 4 features: 1
XofN groups with 5 features: 2
[['f8', 'f6', 'f9', 'f1', 'f5'], ['f4', 'f3', 'f11', 'f2', 'f14'], ['f7', 'f12', 'f10', 'f13']]
